# Ch 13. Embeddings — Solutions

> This notebook contains rigorous solutions and reproducible checks for all five exercises.

## Problem 1

**Problem**: Explain the difference between skip-gram and CBOW, and implement both on a small corpus.

### Rigorous solution

Controlled factor: **objective: Skip-gram center-to-context versus CBOW context-to-center**.  Reported metric: **number and shape of training examples**.  Interpretation and limitation: The same windowed corpus yields multiple center-context pairs for Skip-gram and one aggregated context per center for CBOW; the cell constructs both explicitly.


In [ ]:
import torch
corpus='king queen man woman king man'.split(); sg=[]; cb=[]
for i,w in enumerate(corpus):
 ctx=[corpus[j] for j in range(max(0,i-1),min(len(corpus),i+2)) if j!=i]; sg += [(w,c) for c in ctx]; cb.append((tuple(ctx),w))
print({'skipgram_examples':len(sg),'cbow_examples':len(cb),'skipgram_sample':sg[:3],'cbow_sample':cb[:3]}); assert all(len(x)==2 for x in sg)


## Problem 2

**Problem**: Compare training results for negative sampling with $K = 1, 5, 15$.

### Rigorous solution

Controlled factor: **negative samples K: 1, 5, 15**.  Reported metric: **loss estimate and gradient norm for one positive pair**.  Interpretation and limitation: Matched embeddings and negatives isolate K. More negatives add logistic terms and computation; one stochastic step cannot establish downstream embedding quality.


In [ ]:
import torch
torch.manual_seed(130); center=torch.randn(8,requires_grad=True); pos=torch.randn(8); negatives=torch.randn(15,8)
for K in (1,5,15):
 center.grad=None; loss=-torch.nn.functional.logsigmoid(center@pos)-torch.nn.functional.logsigmoid(-(negatives[:K]@center)).sum(); loss.backward(); print({'K':K,'loss':loss.item(),'center_grad_norm':center.grad.norm().item()})


## Problem 3

**Problem**: Explain the role of GloVe's weighting function $f(x) = (x/x_{\max})^\alpha$ if $x < x_{\max}$ else 1.

### Rigorous solution

Controlled factor: **co-occurrence count x**.  Reported metric: **GloVe weight f(x)**.  Interpretation and limitation: The power law downweights rare/noisy pairs, reaches one at x_max, and caps frequent pairs so they cannot dominate solely by count.


In [ ]:
import torch
xmax=100.; alpha=.75
for x in (1.,5.,20.,100.,500.):
 f=(x/xmax)**alpha if x<xmax else 1.; print({'count':x,'weight':f}); assert 0<f<=1


## Problem 4

**Problem**: Perform word analogy experiments with Word2Vec and analyze the results.

### Rigorous solution

Controlled factor: **analogy vector arithmetic**.  Reported metric: **cosine similarity to every candidate**.  Interpretation and limitation: The synthetic embedding encodes gender and royalty as independent axes, so king-man+woman points exactly to queen. This validates the computation, not real-corpus semantic quality.


In [ ]:
import torch
E={'man':torch.tensor([0.,0.]),'woman':torch.tensor([0.,1.]),'king':torch.tensor([1.,0.]),'queen':torch.tensor([1.,1.]),'apple':torch.tensor([-1.,.2])}; q=E['king']-E['man']+E['woman']; scores={w:torch.nn.functional.cosine_similarity(q,E[w],dim=0).item() for w in E if w not in ('king','man','woman')}; print({'query':'king-man+woman','scores':scores,'answer':max(scores,key=scores.get)}); assert max(scores,key=scores.get)=='queen'


## Problem 5

**Problem**: Explain why $\sqrt{d}$ scaling is applied to transformer embeddings, relating it to the attention formula.

### Rigorous solution

Controlled factor: **embedding scale: 1 versus sqrt(d)**.  Reported metric: **embedding variance and attention-logit variance**.  Interpretation and limitation: If embedding components have variance about 1/d, multiplying by sqrt(d) restores unit component variance before residual/attention projections. The cell measures that second-moment effect.


In [ ]:
import torch
torch.manual_seed(131); d=64; e=torch.randn(20000,d)/d**.5
for scale in (1.,d**.5):
 x=e*scale; q=x@torch.randn(d,d)/d**.5; k=x@torch.randn(d,d)/d**.5; logits=(q*k).sum(1)/d**.5; print({'scale':scale,'embedding_variance':x.var().item(),'logit_variance':logits.var().item()})
